# 03 — Drift Detection

This notebook detects and quantifies data drift between the training set and each production batch.  
We use three complementary methods:
- **KS test** — statistical test for distribution shift in continuous features
- **PSI** (Population Stability Index) — industry-standard drift severity score
- **Evidently AI** — automated drift report with visualisations

**Contents**
1. Setup & data loading
2. KS test — per feature per batch
3. PSI — per feature per batch
4. Label drift (chi-squared test)
5. Evidently drift report
6. Summary & flagged features

## 1. Setup & data loading

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import ks_2samp, chi2_contingency

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

os.chdir(r'C:\Users\baxjo\OneDrive\Documenten\GitHub\datadrift-challenge')
os.makedirs('figures', exist_ok=True)
os.makedirs('reports', exist_ok=True)

In [ ]:
train = pd.read_csv('data/creditcard.csv')

batches = {
    'batch_1': pd.read_csv('data/drift_1.csv'),
    'batch_2': pd.read_csv('data/drift_2.csv'),
    'batch_3': pd.read_csv('data/drift_3.csv'),
    'batch_4': pd.read_csv('data/drift_4.csv'),
    'batch_5': pd.read_csv('data/drift_5.csv'),
}

v_features = [f'V{i}' for i in range(1, 29)]
FEATURES = v_features + ['Amount']

print(f'Training data: {train.shape}')
for name, df in batches.items():
    print(f'{name}: {df.shape}')

## 2. KS Test — per feature per batch

The Kolmogorov-Smirnov test compares two distributions.  
A p-value < 0.05 means the distributions are significantly different (drift detected).  
The KS statistic (0–1) indicates how large the difference is.

In [ ]:
def run_ks_tests(reference, batches, features):
    results = {}
    for batch_name, batch_df in batches.items():
        results[batch_name] = {}
        for feat in features:
            stat, p = ks_2samp(reference[feat], batch_df[feat])
            results[batch_name][feat] = {'ks_stat': round(stat, 4), 'p_value': round(p, 6)}
    return results

ks_results = run_ks_tests(train, batches, FEATURES)

# Build summary dataframes
ks_stats = pd.DataFrame({b: {f: ks_results[b][f]['ks_stat'] for f in FEATURES} for b in batches}).T
ks_pvals = pd.DataFrame({b: {f: ks_results[b][f]['p_value'] for f in FEATURES} for b in batches}).T

print('KS statistics (higher = more drift):')
ks_stats

In [ ]:
# Heatmap of KS statistics
fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(ks_stats, cmap='YlOrRd', ax=ax, linewidths=0.3,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            vmin=0, vmax=0.35)
ax.set_title('KS statistic per feature per batch (higher = more drift)', fontsize=12)
ax.set_xlabel('Feature')
ax.set_ylabel('Batch')
plt.tight_layout()
plt.savefig('figures/drift_ks_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Drift flags: p < 0.05 = drift detected
ks_flags = ks_pvals < 0.05

print('Features flagged as drifted (p < 0.05) per batch:')
for batch in ks_flags.index:
    flagged = ks_flags.columns[ks_flags.loc[batch]].tolist()
    print(f'  {batch}: {len(flagged)} features — {flagged}')

## 3. PSI — Population Stability Index

PSI measures how much a feature distribution has shifted between reference and production.  

| PSI value | Interpretation |
|-----------|---------------|
| < 0.1     | No significant drift |
| 0.1 – 0.2 | Moderate drift — monitor |
| > 0.2     | Significant drift — action required |

In [ ]:
def calculate_psi(reference, production, bins=10):
    """Calculate Population Stability Index between two arrays."""
    # Create bins based on reference distribution
    breakpoints = np.linspace(reference.min(), reference.max(), bins + 1)
    breakpoints[0]  = -np.inf
    breakpoints[-1] =  np.inf

    ref_counts  = np.histogram(reference,  bins=breakpoints)[0]
    prod_counts = np.histogram(production, bins=breakpoints)[0]

    # Convert to proportions, avoid division by zero
    ref_pct  = np.where(ref_counts  == 0, 0.0001, ref_counts  / len(reference))
    prod_pct = np.where(prod_counts == 0, 0.0001, prod_counts / len(production))

    psi = np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct))
    return round(psi, 4)

# Calculate PSI for all features and batches
psi_results = {}
for batch_name, batch_df in batches.items():
    psi_results[batch_name] = {}
    for feat in FEATURES:
        psi_results[batch_name][feat] = calculate_psi(train[feat].values, batch_df[feat].values)

psi_df = pd.DataFrame(psi_results).T

print('PSI values per feature per batch:')
psi_df

In [ ]:
# PSI heatmap with threshold indicators
fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(psi_df, cmap='YlOrRd', ax=ax, linewidths=0.3,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            vmin=0, vmax=0.5)
ax.set_title('PSI per feature per batch  |  >0.2 = significant drift', fontsize=12)
ax.set_xlabel('Feature')
ax.set_ylabel('Batch')
plt.tight_layout()
plt.savefig('figures/drift_psi_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# PSI flags: > 0.2 = significant drift
psi_flags = psi_df > 0.2

print('Features with PSI > 0.2 (significant drift) per batch:')
for batch in psi_flags.index:
    flagged = psi_flags.columns[psi_flags.loc[batch]].tolist()
    print(f'  {batch}: {len(flagged)} features — {flagged}')

print()
# Overall PSI per batch (mean across features)
print('Mean PSI per batch:')
print(psi_df.mean(axis=1).round(4))

In [ ]:
# Distribution comparison for top drifted features
# Pick top 4 features by PSI in worst batch (batch_5)
top_features = psi_df.loc['batch_5'].nlargest(4).index.tolist()

fig, axes = plt.subplots(len(top_features), 5, figsize=(18, 3.5 * len(top_features)))

batch_colors = ['#e07b54', '#5b8db8', '#6aab7a', '#b87eb8', '#b8a84a']

for row, feat in enumerate(top_features):
    ref_vals = train[feat].clip(
        lower=train[feat].quantile(0.01),
        upper=train[feat].quantile(0.99)
    )
    for col, (batch_name, batch_df) in enumerate(batches.items()):
        ax = axes[row, col]
        prod_vals = batch_df[feat].clip(
            lower=train[feat].quantile(0.01),
            upper=train[feat].quantile(0.99)
        )
        ax.hist(ref_vals,  bins=40, alpha=0.5, color='steelblue', label='Train', density=True)
        ax.hist(prod_vals, bins=40, alpha=0.5, color=batch_colors[col], label=batch_name, density=True)
        psi_val = psi_df.loc[batch_name, feat]
        ks_val  = ks_stats.loc[batch_name, feat]
        ax.set_title(f'{feat} | {batch_name}\nPSI={psi_val:.2f}  KS={ks_val:.2f}', fontsize=9)
        ax.legend(fontsize=7)
        ax.set_yticks([])

plt.suptitle('Top drifted features — train vs production batches', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('figures/drift_top_features.png', bbox_inches='tight')
plt.show()

## 4. Label drift — chi-squared test

Has the fraud rate itself changed significantly between training and each batch?

In [ ]:
label_results = []

for batch_name, batch_df in batches.items():
    # Contingency table: [legitimate, fraud] for train vs batch
    ref_counts  = train['Class'].value_counts().sort_index().values
    prod_counts = batch_df['Class'].value_counts().sort_index().values

    # Normalize to same size for fair comparison
    ref_norm  = ref_counts  / ref_counts.sum()
    prod_norm = prod_counts / prod_counts.sum()

    contingency = np.array([ref_counts, prod_counts])
    chi2, p, dof, expected = chi2_contingency(contingency)

    label_results.append({
        'batch': batch_name,
        'train_fraud_rate_%': round(train['Class'].mean() * 100, 4),
        'batch_fraud_rate_%': round(batch_df['Class'].mean() * 100, 4),
        'chi2_stat': round(chi2, 2),
        'p_value': round(p, 6),
        'drift_detected': p < 0.05
    })

label_df = pd.DataFrame(label_results)
print('Label drift results:')
label_df

In [ ]:
# Visualise fraud rate over batches
fig, ax = plt.subplots(figsize=(10, 5))

batch_names  = label_df['batch'].tolist()
fraud_rates  = label_df['batch_fraud_rate_%'].tolist()
train_rate   = label_df['train_fraud_rate_%'].iloc[0]
bar_colors   = ['tomato' if d else 'steelblue' for d in label_df['drift_detected']]

bars = ax.bar(batch_names, fraud_rates, color=bar_colors, alpha=0.85)
ax.axhline(train_rate, color='navy', linestyle='--', linewidth=1.5, label=f'Train baseline ({train_rate:.2f}%)')

for bar, rate in zip(bars, fraud_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{rate:.2f}%', ha='center', fontsize=10)

red_patch  = mpatches.Patch(color='tomato',    label='Drift detected (p < 0.05)')
blue_patch = mpatches.Patch(color='steelblue', label='No significant drift')
ax.legend(handles=[red_patch, blue_patch, ax.get_lines()[0]])
ax.set_title('Label drift — fraud rate per batch vs training baseline')
ax.set_ylabel('Fraud rate (%)')
plt.tight_layout()
plt.savefig('figures/drift_label.png', bbox_inches='tight')
plt.show()

## 5. Evidently drift report

In [ ]:
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset, DataQualityPreset
    EVIDENTLY_AVAILABLE = True
    print('Evidently available — generating full report')
except ImportError:
    EVIDENTLY_AVAILABLE = False
    print('Evidently not installed. Run: pip install evidently')
    print('Skipping to summary section.')

In [ ]:
if EVIDENTLY_AVAILABLE:
    # Generate one Evidently report per batch
    report_features = v_features + ['Amount']
    
    for batch_name, batch_df in batches.items():
        ref  = train[report_features + ['Class']].sample(5000, random_state=42)  # sample for speed
        curr = batch_df[report_features + ['Class']]
        
        report = Report(metrics=[DataDriftPreset()])
        report.run(reference_data=ref, current_data=curr)
        report.save_html(f'reports/evidently_{batch_name}.html')
        print(f'Saved: reports/evidently_{batch_name}.html')
else:
    print('Skipped — install evidently first with:')
    print('  pip install evidently')

## 6. Summary & flagged features

In [ ]:
# Combined drift summary per batch
summary_rows = []

for batch_name in batches:
    ks_flagged  = ks_flags.columns[ks_flags.loc[batch_name]].tolist()
    psi_flagged = psi_flags.columns[psi_flags.loc[batch_name]].tolist()
    both        = list(set(ks_flagged) & set(psi_flagged))  # flagged by both methods
    label_drift = label_df[label_df['batch'] == batch_name]['drift_detected'].values[0]

    summary_rows.append({
        'batch': batch_name,
        'ks_flagged_count': len(ks_flagged),
        'psi_flagged_count': len(psi_flagged),
        'flagged_by_both': len(both),
        'top_drifted_features': ', '.join(psi_df.loc[batch_name].nlargest(3).index.tolist()),
        'label_drift': label_drift,
        'fraud_rate_%': round(batches[batch_name]['Class'].mean() * 100, 4),
        'mean_psi': round(psi_df.loc[batch_name].mean(), 4)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('reports/drift_summary.csv', index=False)
print('Drift summary saved to reports/drift_summary.csv')
summary_df

In [ ]:
# Overall drift severity plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean PSI per batch
axes[0].bar(summary_df['batch'], summary_df['mean_psi'],
            color=['tomato' if p > 0.2 else 'steelblue' for p in summary_df['mean_psi']])
axes[0].axhline(0.2, color='red', linestyle='--', linewidth=1, label='PSI > 0.2 threshold')
axes[0].axhline(0.1, color='orange', linestyle='--', linewidth=1, label='PSI > 0.1 threshold')
axes[0].set_title('Mean PSI per batch')
axes[0].set_ylabel('Mean PSI')
axes[0].legend(fontsize=8)
for i, v in enumerate(summary_df['mean_psi']):
    axes[0].text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)

# Number of drifted features per batch
x = np.arange(len(summary_df))
width = 0.35
axes[1].bar(x - width/2, summary_df['ks_flagged_count'],  width, label='KS flagged',  color='steelblue', alpha=0.8)
axes[1].bar(x + width/2, summary_df['psi_flagged_count'], width, label='PSI flagged', color='tomato',    alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df['batch'])
axes[1].set_title('Number of drifted features per batch')
axes[1].set_ylabel('Feature count')
axes[1].legend()

plt.suptitle('Drift severity overview', fontsize=13)
plt.tight_layout()
plt.savefig('figures/drift_severity_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=== DRIFT DETECTION SUMMARY ===')
print()
for _, row in summary_df.iterrows():
    print(f"{row['batch']}")
    print(f"  Fraud rate     : {row['fraud_rate_%']}%")
    print(f"  Label drift    : {'YES' if row['label_drift'] else 'no'}")
    print(f"  KS flagged     : {row['ks_flagged_count']} features")
    print(f"  PSI flagged    : {row['psi_flagged_count']} features")
    print(f"  Top features   : {row['top_drifted_features']}")
    print(f"  Mean PSI       : {row['mean_psi']}")
    print()